In [9]:
from typing import TypedDict, Annotated, Literal
import sqlite3
import requests

from dotenv import load_dotenv
from pydantic import BaseModel

from langchain_ollama import ChatOllama
from langchain_core.messages import BaseMessage, SystemMessage,HumanMessage
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_ollama import OllamaEmbeddings
load_dotenv()
import os
from langgraph.graph import MessagesState

In [10]:
llm = ChatOllama(
    model="llama3.2",
    temperature=0,
    base_url="http://172.31.0.1:11434"
)


In [11]:
def call_model(state:MessagesState):
    response=llm.invoke(state["messages"])
    return {"messages": [response]}

In [12]:
from langgraph.checkpoint.memory import InMemorySaver
checkpoint_saver=InMemorySaver()



In [13]:
builder=StateGraph(MessagesState)
builder.add_node("call_model",call_model)

builder.add_edge(START,"call_model")

graph=builder.compile(checkpointer=checkpoint_saver)

In [14]:
config={"configurable":{"thread_id":"1"}}

In [15]:
graph.invoke({"messages":[{"role":"user","content":"Hello my name is sanjay"}]},config=config)


{'messages': [HumanMessage(content='Hello my name is sanjay', additional_kwargs={}, response_metadata={}, id='d66f38ae-c628-464a-aef6-85dcf8fa2472'),
  AIMessage(content="Namaste Sanjay! It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-08-24T15:23:51.3908833Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6560567000, 'load_duration': 4344229500, 'prompt_eval_count': 31, 'prompt_eval_duration': 343020000, 'eval_count': 28, 'eval_duration': 1860889000, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--01a0345e-d33b-70f2-b1bf-6c6e90c9217e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 31, 'output_tokens': 28, 'total_tokens': 59})]}

In [16]:
graph.invoke({"messages":[{"role":"user","content":"what is my name?"}]},config=config)

{'messages': [HumanMessage(content='Hello my name is sanjay', additional_kwargs={}, response_metadata={}, id='d66f38ae-c628-464a-aef6-85dcf8fa2472'),
  AIMessage(content="Namaste Sanjay! It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-08-24T15:23:51.3908833Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6560567000, 'load_duration': 4344229500, 'prompt_eval_count': 31, 'prompt_eval_duration': 343020000, 'eval_count': 28, 'eval_duration': 1860889000, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--01a0345e-d33b-70f2-b1bf-6c6e90c9217e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 31, 'output_tokens': 28, 'total_tokens': 59}),
  HumanMessage(content='what is my name?', additional_kwargs={}, response_metadata={}, id='b1fbac26-8475-4f03-8edf-0ba2764f5571'),
  AIMessage(content='Your name is San

In [ ]:
snap=graph.get_state(config=config)
vals=snap.values
for m in vals.get("messages",[]):
    print("-",type(m).__name__,":",m.content)

- HumanMessage : Hello my name is sanjay
- AIMessage : Namaste Sanjay! It's nice to meet you. Is there something I can help you with or would you like to chat?
- HumanMessage : what is my name?
- AIMessage : Your name is Sanjay, as we established earlier! How can I assist you today, Sanjay?
